# ❄️ Boston Snowiest Streets — Data Pipeline
### Winter 2025–2026

This notebook builds all data files needed for the interactive Observable visualization.

**Outputs:**
- `boston_streets_snow.geojson` — streets with seasonal + per-storm snowfall
- `boston_storms.json` — major storm events
- `boston_historical.json` — winter totals 1980–2026
- `boston_neighborhoods.geojson` — neighborhood boundaries + snowfall
- `boston_plow_routes.geojson` — plow route coverage

**Instructions:**
1. Upload `DailyPrecipReports_MA_2025-10-01_1.csv` to Colab files panel
2. Paste your NOAA API key in Cell 3
3. Run all cells top to bottom
4. Download all files from the `outputs/` folder

In [53]:
# CELL 1 — Install dependencies
!pip install osmnx geopandas scipy shapely requests pandas numpy -q

In [54]:
# CELL 2 — Imports
import os, json, requests
import pandas as pd
import numpy as np
import geopandas as gpd
import osmnx as ox
from scipy.spatial import cKDTree
from shapely.geometry import Point
from collections import defaultdict
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

os.makedirs('outputs', exist_ok=True)
print('Imports ready')

Imports ready


In [55]:
# CELL 3 — CONFIGURATION  ← paste your NOAA key here
NOAA_API_KEY   = 'NDJeXTCmRuSQrLzNQVPNZhvDLsMrAytO'
COCORAHS_FILE  = 'DailyPrecipReports_MA_2025-10-01_1.csv'

BOSTON_BBOX = dict(lat_min=42.22, lat_max=42.55, lon_min=-71.35, lon_max=-70.85)
SEASON_START = '2025-10-01'
SEASON_END   = '2026-03-31'
IDW_POWER    = 2

print('Config set')

Config set


## 1. Load CoCoRaHS Data

In [37]:
# CELL 4 — Load and clean CoCoRaHS daily reports
df_raw = pd.read_csv(COCORAHS_FILE)
df_raw.columns = df_raw.columns.str.strip()
df_raw = df_raw.apply(lambda c: c.str.strip() if c.dtype == 'object' else c)

df_raw['ObservationDate'] = pd.to_datetime(df_raw['ObservationDate'])
df_raw['Latitude']        = pd.to_numeric(df_raw['Latitude'],        errors='coerce')
df_raw['Longitude']       = pd.to_numeric(df_raw['Longitude'],       errors='coerce')
df_raw['NewSnowDepth']    = pd.to_numeric(df_raw['NewSnowDepth'],    errors='coerce').fillna(0)

df = df_raw[(df_raw['ObservationDate'] >= SEASON_START) &
            (df_raw['ObservationDate'] <= SEASON_END)].copy()

mask = (df['Latitude'].between(BOSTON_BBOX['lat_min'], BOSTON_BBOX['lat_max']) &
        df['Longitude'].between(BOSTON_BBOX['lon_min'], BOSTON_BBOX['lon_max']))
df_boston = df[mask].copy()

print(f'Total MA records : {len(df):,}')
print(f'Boston records   : {len(df_boston):,}')
print(f'Boston stations  : {df_boston["StationNumber"].nunique()}')
print(f'Date range       : {df_boston["ObservationDate"].min().date()} to {df_boston["ObservationDate"].max().date()}')

Total MA records : 47,516
Boston records   : 5,377
Boston stations  : 44
Date range       : 2025-10-01 to 2026-03-08


In [38]:
# CELL 5 — Seasonal totals per station
station_totals = (
    df_boston
    .groupby(['StationNumber','StationName','Latitude','Longitude'])['NewSnowDepth']
    .sum().reset_index()
    .rename(columns={'NewSnowDepth':'seasonal_snow_in'})
)
counts = df_boston.groupby('StationNumber').size().reset_index(name='ndays')
station_totals = station_totals.merge(counts, on='StationNumber')
station_totals = station_totals[station_totals['ndays'] >= 60]

print(f'Quality stations: {len(station_totals)}')
print(station_totals[['StationName','seasonal_snow_in']]
      .sort_values('seasonal_snow_in', ascending=False).head(10).to_string(index=False))

Quality stations: 34
      StationName  seasonal_snow_in
   Quincy 1.5 SSE              89.2
    Chelsea 0.8 N              87.7
Winchester 1.2 SE              82.4
     Milton 1.3 N              81.8
 Winchester 0.8 W              81.2
Winchester 0.7 SE              78.2
  Watertown 1.1 W              70.1
  Carlisle 2.0 SE              68.2
   Boston 2.6 WSW              67.3
   Natick 1.9 NNE              67.1


## 2. Detect Major Storms

In [39]:
# CELL 6 — Identify storm events
daily_avg = (
    df_boston[df_boston['NewSnowDepth'] > 0]
    .groupby('ObservationDate')
    .agg(avg_snow=('NewSnowDepth','mean'), max_snow=('NewSnowDepth','max'),
         n_stations=('StationNumber','nunique'))
    .reset_index()
)
storm_days = daily_avg[(daily_avg['avg_snow'] > 1.0) & (daily_avg['n_stations'] >= 5)].sort_values('ObservationDate')

storms = []
current_storm = None
for _, row in storm_days.iterrows():
    if current_storm is None:
        current_storm = {'start': row['ObservationDate'], 'end': row['ObservationDate'], 'days': [row]}
    else:
        gap = (row['ObservationDate'] - current_storm['end']).days
        if gap <= 2:
            current_storm['end'] = row['ObservationDate']
            current_storm['days'].append(row)
        else:
            storms.append(current_storm)
            current_storm = {'start': row['ObservationDate'], 'end': row['ObservationDate'], 'days': [row]}
if current_storm:
    storms.append(current_storm)

storm_records = []
for i, s in enumerate(storms):
    days_df = pd.DataFrame(s['days'])
    start, end = s['start'], s['end']
    label = start.strftime('%b %-d') if start == end else f"{start.strftime('%b %-d')}-{end.strftime('%-d, %Y')}"
    storm_records.append({
        'id': f'storm_{i+1:02d}', 'start_date': start.strftime('%Y-%m-%d'),
        'end_date': end.strftime('%Y-%m-%d'), 'label': label,
        'avg_snow_in': round(days_df['avg_snow'].sum(), 1),
        'peak_snow_in': round(days_df['max_snow'].max(), 1),
        'n_days': len(s['days'])
    })

storm_records = sorted(storm_records, key=lambda x: x['avg_snow_in'], reverse=True)
print(f'{len(storm_records)} storms detected:')
for s in storm_records:
    print(f"  {s['label']:30s} {s['avg_snow_in']:5.1f}in avg | {s['peak_snow_in']:5.1f}in peak")

7 storms detected:
  Jan 26-27, 2026                 17.8in avg |  18.2in peak
  Feb 21-24, 2026                 16.6in avg |  21.0in peak
  Feb 7-8, 2026                    5.3in avg |   8.0in peak
  Jan 19                           4.7in avg |   6.4in peak
  Dec 27                           2.6in avg |   3.5in peak
  Feb 11                           1.6in avg |   2.5in peak
  Mar 4                            1.4in avg |   2.5in peak


In [40]:
# CELL 7 — Per-storm station snowfall
for storm in storm_records:
    mask = ((df_boston['ObservationDate'] >= storm['start_date']) &
            (df_boston['ObservationDate'] <= storm['end_date']))
    s_df = (df_boston[mask]
            .groupby(['StationNumber','StationName','Latitude','Longitude'])['NewSnowDepth']
            .sum().reset_index())
    s_df = s_df[s_df['NewSnowDepth'] > 0]
    storm['stations'] = [
        {'id': r['StationNumber'], 'name': r['StationName'],
         'lat': round(r['Latitude'],6), 'lon': round(r['Longitude'],6),
         'snow': round(r['NewSnowDepth'],1)}
        for _, r in s_df.iterrows()
    ]
print('Per-storm station data attached')

Per-storm station data attached


## 3. NOAA Historical Data (1980–2026)

In [57]:
# CELL 8 — Fetch historical winters from NOAA GHCN + CoCoRaHS (take higher)
NOAA_BASE  = 'https://www.ncdc.noaa.gov/cdo-web/api/v2/data'
STATION_ID = 'GHCND:USW00014739'   # Boston Logan Airport
HEADERS    = {'token': NOAA_API_KEY}

def fetch_noaa_snow(year_start, year_end, max_retries=3):
    params = {
        'datasetid': 'GHCND', 'datatypeid': 'SNOW', 'stationid': STATION_ID,
        'startdate': f'{year_start}-10-01', 'enddate': f'{year_end}-03-31',
        'limit': 1000, 'units': 'metric'
    }
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.get(NOAA_BASE, headers=HEADERS, params=params, timeout=20)
            if r.status_code == 200:
                data = r.json()
                if 'results' not in data:
                    return 0.0
                total_mm = sum(d['value'] for d in data['results'] if d['value'] > 0)
                return round(total_mm / 25.4, 1)
            elif r.status_code == 503:
                print(f'  503 for {year_start}-{year_end}, attempt {attempt}/{max_retries}, waiting {attempt*5}s...')
                time.sleep(attempt * 5)
            else:
                print(f'  API error {r.status_code} for {year_start}-{year_end}')
                return None
        except Exception as e:
            print(f'  Exception for {year_start}-{year_end} attempt {attempt}: {e}')
            time.sleep(attempt * 5)
    print(f'  Failed after {max_retries} retries for {year_start}-{year_end}')
    return None

def fetch_cocorahs_snow(year_start, year_end, df):
    # Filter to Boston metro area only
    if 'Latitude' in df.columns and 'Longitude' in df.columns:
        df = df[
            (df['Latitude'].between(42.1, 42.6)) &
            (df['Longitude'].between(-71.4, -70.8))
        ].copy()

    mask = (
        (df['ObsDate'] >= f'{year_start}-10-01') &
        (df['ObsDate'] <= f'{year_end}-03-31') &
        (df['NewSnowDepth'] > 0)
    )
    subset = df[mask]
    if subset.empty:
        return None
    daily = subset.groupby('ObsDate')['NewSnowDepth'].mean()
    return round(daily.sum(), 1)

# Load CoCoRaHS CSV
import glob
csv_files = glob.glob('/content/*.csv') + glob.glob('*.csv')
cocorahs_file = next((f for f in csv_files if 'Daily' in f or 'Precip' in f), None)

if cocorahs_file:
    df_daily = pd.read_csv(cocorahs_file)
    df_daily.columns = df_daily.columns.str.strip()
    date_col = next((c for c in df_daily.columns if 'date' in c.lower()), None)
    if date_col and date_col != 'ObsDate':
        df_daily = df_daily.rename(columns={date_col: 'ObsDate'})
    snow_col = next((c for c in df_daily.columns if 'snow' in c.lower()), None)
    if snow_col and snow_col != 'NewSnowDepth':
        df_daily = df_daily.rename(columns={snow_col: 'NewSnowDepth'})
    df_daily['NewSnowDepth'] = pd.to_numeric(df_daily['NewSnowDepth'], errors='coerce').fillna(0)
    print(f'CoCoRaHS loaded: {len(df_daily):,} rows')
else:
    print('CoCoRaHS CSV not found — NOAA only')
    df_daily = pd.DataFrame(columns=['ObsDate', 'NewSnowDepth'])

df_hist = df_daily.copy()
df_hist['ObsDate'] = df_hist['ObsDate'].astype(str).str[:10]

historical = []
print('Fetching historical winters (Logan + CoCoRaHS, taking higher)...')

for start_year in range(1980, 2026):
    end_year = start_year + 1
    label    = f'{start_year}-{str(end_year)[2:]}'

    noaa_val     = fetch_noaa_snow(start_year, end_year)
    cocorahs_val = fetch_cocorahs_snow(start_year, end_year, df_hist)

    candidates = {k: v for k, v in {'NOAA GHCN': noaa_val, 'CoCoRaHS': cocorahs_val}.items() if v is not None}
    if not candidates:
        print(f'  {label}: no data')
        continue

    best_source = max(candidates, key=candidates.get)
    best_val    = candidates[best_source]
    is_current  = (start_year == 2025)

    historical.append({
        'winter': label, 'start_year': start_year, 'snow_in': best_val,
        'source': best_source, 'is_current': is_current
    })
    tag = f'[{best_source}]' if len(candidates) > 1 else f'[{best_source} only]'
    print(f'  {label}: {best_val}in  {tag}')

# Rankings
ranked = sorted(historical, key=lambda x: x['snow_in'], reverse=True)
for i, h in enumerate(ranked):
    h['rank'] = i + 1

snow_vals = [h['snow_in'] for h in historical]
mean_snow = round(float(np.mean(snow_vals)), 1)
for h in historical:
    h['pct_above_avg'] = round((h['snow_in'] - mean_snow) / mean_snow * 100, 1)
    if   h['snow_in'] >= mean_snow * 1.5: h['tier'] = 'historic'
    elif h['snow_in'] >= mean_snow * 1.1: h['tier'] = 'above_avg'
    elif h['snow_in'] >= mean_snow * 0.7: h['tier'] = 'average'
    else:                                  h['tier'] = 'below_avg'
    if h['start_year'] == 2025:           h['is_current'] = True

print(f'\nHistorical data: {len(historical)} winters')
print(f'Season average : {mean_snow} inches')

CoCoRaHS loaded: 47,516 rows
Fetching historical winters (Logan + CoCoRaHS, taking higher)...
  1980-81: 22.3in  [NOAA GHCN only]
  1981-82: 48.5in  [NOAA GHCN only]
  1982-83: 32.8in  [NOAA GHCN only]
  1983-84: 43.0in  [NOAA GHCN only]
  1984-85: 24.8in  [NOAA GHCN only]
  1985-86: 18.1in  [NOAA GHCN only]
  Exception for 1986-1987 attempt 1: HTTPSConnectionPool(host='www.ncdc.noaa.gov', port=443): Read timed out. (read timeout=20)
  1986-87: 38.4in  [NOAA GHCN only]
  1987-88: 52.6in  [NOAA GHCN only]
  Exception for 1988-1989 attempt 1: HTTPSConnectionPool(host='www.ncdc.noaa.gov', port=443): Read timed out. (read timeout=20)
  1988-89: 15.2in  [NOAA GHCN only]
  1989-90: 38.7in  [NOAA GHCN only]
  1990-91: 19.1in  [NOAA GHCN only]
  1991-92: 21.1in  [NOAA GHCN only]
  1992-93: 81.9in  [NOAA GHCN only]
  1993-94: 96.3in  [NOAA GHCN only]
  1994-95: 15.1in  [NOAA GHCN only]
  Exception for 1995-1996 attempt 1: HTTPSConnectionPool(host='www.ncdc.noaa.gov', port=443): Read timed out. 

## 4. Boston Street Network

In [69]:
# CELL 9 — Download Boston street network via OSMnx
print('Downloading Boston street network (~2 min)...')

PLACES = [
    'Boston, Massachusetts, USA',
    'Cambridge, Massachusetts, USA',
    'Brookline, Massachusetts, USA',
    'Newton, Massachusetts, USA',]

import networkx as nx
graphs = []
for place in PLACES:
    try:
        g = ox.graph_from_place(place, network_type='drive', retain_all=False)
        graphs.append(g)
        print(f'  OK  {place}')
    except Exception as e:
        print(f'  --  {place}: {e}')

G = nx.compose_all(graphs)
print(f'\nTotal nodes: {G.number_of_nodes():,} | edges: {G.number_of_edges():,}')
nodes, edges = ox.graph_to_gdfs(G)
edges = edges.reset_index()

keep = ['osmid','name','highway','length','geometry']
edges = edges[[c for c in keep if c in edges.columns]].copy()

edges['midpoint']    = edges['geometry'].interpolate(0.5, normalized=True)
edges['mid_lat']     = edges['midpoint'].y
edges['mid_lon']     = edges['midpoint'].x
edges['street_name'] = edges['name'].apply(
    lambda x: x[0] if isinstance(x, list) else (x if isinstance(x, str) else 'Unnamed'))

def road_type(hw):
    if isinstance(hw, list): hw = hw[0]
    if not isinstance(hw, str): return 'other'
    if hw in ['motorway','trunk','primary']:        return 'major'
    if hw in ['secondary','tertiary']:              return 'secondary'
    if hw in ['residential','living_street']:       return 'residential'
    return 'other'

edges['road_type'] = edges['highway'].apply(road_type)
print(f'Street segments: {len(edges):,}')
print(edges['road_type'].value_counts())

  OK  Boston, Massachusetts, USA
  OK  Cambridge, Massachusetts, USA
  OK  Brookline, Massachusetts, USA
  OK  Newton, Massachusetts, USA

Total nodes: 17,608 | edges: 41,269
Street segments: 41,269
road_type
residential    24455
secondary      10237
major           5186
other           1391
Name: count, dtype: int64


## 5. Boston Neighborhoods & Plow Routes

In [70]:
# CELL 10 — Neighborhood boundaries from Boston Open Data
NEIGHBORHOODS_URL = (
    'https://bostonopendata-boston.opendata.arcgis.com/datasets/'
    'boston::boston-neighborhoods.geojson'
)
try:
    r = requests.get(NEIGHBORHOODS_URL, timeout=30)
    neighborhoods = gpd.GeoDataFrame.from_features(r.json()['features'], crs='EPSG:4326')
    name_col = next((c for c in ['Name','NAME','name','Neighborhood'] if c in neighborhoods.columns), None)
    print(f'Neighborhoods loaded: {len(neighborhoods)} | name col: {name_col}')
except Exception as e:
    print(f'Neighborhoods unavailable: {e}')
    neighborhoods = None
    name_col = None

Neighborhoods loaded: 25 | name col: Name


In [71]:
# CELL 11 — Boston plow routes from open data
PLOW_URLS = [
    'https://bostonopendata-boston.opendata.arcgis.com/datasets/boston::snow-emergency-routes.geojson',
    'https://opendata.arcgis.com/datasets/9a3a8c427add450eaf45a470245680fc_0.geojson'
]
plow_routes = None
for url in PLOW_URLS:
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 200 and r.json().get('features'):
            plow_routes = gpd.GeoDataFrame.from_features(r.json()['features'], crs='EPSG:4326')
            print(f'Plow routes loaded: {len(plow_routes)} segments from {url}')
            break
    except Exception as e:
        print(f'Tried: {e}')

if plow_routes is None:
    print('Plow route API unavailable — deriving from road classification')
    print('  Major/secondary = Priority 1-2 | Residential = Priority 3')

Plow routes loaded: 737 segments from https://bostonopendata-boston.opendata.arcgis.com/datasets/boston::snow-emergency-routes.geojson


## 6. Spatial Interpolation (IDW)

In [72]:
# CELL 12 — IDW interpolation
def idw_interpolate(query_lats, query_lons, station_df, value_col, power=2, k=8):
    coords   = station_df[['Latitude','Longitude']].values
    queries  = np.column_stack([query_lats, query_lons])
    values   = station_df[value_col].values
    tree     = cKDTree(coords)
    k_use    = min(k, len(coords))
    dists, idxs = tree.query(queries, k=k_use)
    dists    = np.where(dists == 0, 1e-10, dists)
    weights  = 1.0 / (dists ** power)
    weights /= weights.sum(axis=1, keepdims=True)
    return np.round((weights * values[idxs]).sum(axis=1), 1)

# Seasonal
print('IDW: seasonal totals...')
edges['snow_seasonal'] = idw_interpolate(
    edges['mid_lat'].values, edges['mid_lon'].values,
    station_totals, 'seasonal_snow_in')

# Per-storm
print('IDW: per-storm...')
for storm in storm_records:
    col = f"snow_{storm['id']}"
    if not storm['stations']:
        edges[col] = 0.0
        continue
    s_df = pd.DataFrame(storm['stations']).rename(
        columns={'lat':'Latitude','lon':'Longitude','snow':'snow_in'})
    edges[col] = idw_interpolate(
        edges['mid_lat'].values, edges['mid_lon'].values, s_df, 'snow_in')
    print(f"  {storm['label']:28s} street avg {edges[col].mean():.1f}in")

print(f'\nSeasonal range: {edges["snow_seasonal"].min():.1f} - {edges["snow_seasonal"].max():.1f} inches')

IDW: seasonal totals...
IDW: per-storm...
  Jan 26-27, 2026              street avg 35.5in
  Feb 21-24, 2026              street avg 14.2in
  Feb 7-8, 2026                street avg 4.9in
  Jan 19                       street avg 9.5in
  Dec 27                       street avg 2.8in
  Feb 11                       street avg 1.3in
  Mar 4                        street avg 1.0in

Seasonal range: 0.0 - 77.6 inches


In [73]:
# CELL 13 — Assign plow priority
if isinstance(plow_routes, gpd.GeoDataFrame):
    edges_gdf = edges.set_geometry('geometry').set_crs('EPSG:4326')
    plow_gdf  = plow_routes.set_crs('EPSG:4326', allow_override=True)
    joined    = gpd.sjoin(edges_gdf, plow_gdf[['geometry']], how='left', predicate='intersects')

    # A street may match multiple plow routes — keep one row per street
    is_plow = joined['index_right'].notna()
    is_plow = is_plow.groupby(is_plow.index).any()          # deduplicate
    edges['is_plow_route'] = is_plow.reindex(edges.index, fill_value=False)
else:
    edges['is_plow_route'] = edges['road_type'].isin(['major','secondary'])

def plow_priority(row):
    if row['is_plow_route'] or row['road_type'] == 'major': return 1
    if row['road_type'] == 'secondary':                      return 2
    return 3

edges['plow_priority'] = edges.apply(plow_priority, axis=1)
print('Plow priorities:')
print(edges['plow_priority'].value_counts())

Plow priorities:
plow_priority
3    22928
1    10564
2     7777
Name: count, dtype: int64


In [74]:
# CELL 14 — Assign neighborhood labels
if neighborhoods is not None and name_col:
    midpoints_gdf = gpd.GeoDataFrame(
        edges[['mid_lat','mid_lon']],
        geometry=gpd.points_from_xy(edges['mid_lon'], edges['mid_lat']),
        crs='EPSG:4326')
    nb_proj = neighborhoods[[name_col,'geometry']].set_crs('EPSG:4326', allow_override=True)
    joined  = gpd.sjoin(midpoints_gdf, nb_proj, how='left', predicate='within')
    edges['neighborhood'] = joined[name_col].values
    edges['neighborhood'] = edges['neighborhood'].fillna('Other')
    print('Neighborhoods assigned:')
    print(edges['neighborhood'].value_counts().head(10))
else:
    edges['neighborhood'] = 'Boston'
    print('All streets labeled Boston (neighborhood data unavailable)')

Neighborhoods assigned:
neighborhood
Other            15234
Dorchester        4589
West Roxbury      2471
Roxbury           2194
Hyde Park         2133
Jamaica Plain     1952
Brighton          1660
South Boston      1551
Roslindale        1450
East Boston       1336
Name: count, dtype: int64


## 7. Export All Files

In [75]:
# CELL 15 — Export streets GeoJSON
storm_cols   = [f"snow_{s['id']}" for s in storm_records]
export_cols  = ['street_name','road_type','neighborhood','snow_seasonal',
                'plow_priority','is_plow_route','length','geometry'] + storm_cols

streets_out = edges[[c for c in export_cols if c in edges.columns]].copy()
streets_out = streets_out.set_geometry('geometry').set_crs('EPSG:4326')
for col in ['snow_seasonal','length'] + storm_cols:
    if col in streets_out.columns:
        streets_out[col] = streets_out[col].round(1)
streets_out['is_plow_route'] = streets_out['is_plow_route'].astype(bool)

path = 'outputs/boston_streets_snow.geojson'
streets_out.to_file(path, driver='GeoJSON')
print(f'Streets: {len(streets_out):,} segments | {os.path.getsize(path)/1e6:.1f} MB')

Streets: 41,269 segments | 25.4 MB


In [76]:
# CELL 16 — Export storms JSON
storms_export = []
for s in storm_records:
    rec = {k: v for k, v in s.items() if k != 'stations'}
    rec['snow_field'] = f"snow_{s['id']}"
    storms_export.append(rec)

with open('outputs/boston_storms.json','w') as f:
    json.dump({
        'generated': datetime.now().strftime('%Y-%m-%d'),
        'season': '2025-2026',
        'storms': storms_export,
        'stations': {'seasonal': [
            {'id': r['StationNumber'], 'name': r['StationName'],
             'lat': round(r['Latitude'],6), 'lon': round(r['Longitude'],6),
             'snow': round(r['seasonal_snow_in'],1)}
            for _, r in station_totals.iterrows()
        ]}
    }, f, indent=2)
print(f'Storms: {len(storms_export)} events exported')

Storms: 7 events exported


In [77]:
# CELL 17 — Export historical JSON
with open('outputs/boston_historical.json','w') as f:
    json.dump({
        'generated':   datetime.now().strftime('%Y-%m-%d'),
        'station':     'Boston Logan Airport (USW00014739)',
        'mean_snow':   mean_snow,
        'median_snow': round(float(np.median([h['snow_in'] for h in historical])), 1),
        'years':       historical
    }, f, indent=2)
current = next(h for h in historical if h.get('is_current'))
print(f'Historical: {len(historical)} winters | 2025-2026 rank #{current["rank"]}')

Historical: 46 winters | 2025-2026 rank #10


In [78]:
# CELL 18 — Export neighborhoods GeoJSON
if neighborhoods is not None and name_col:
    nb_snow = (edges[edges['neighborhood'] != 'Other']
               .groupby('neighborhood')['snow_seasonal']
               .agg(['mean','max','count']).reset_index()
               .rename(columns={'mean':'avg_snow','max':'max_snow','count':'n_streets'}))
    nb_snow[['avg_snow','max_snow']] = nb_snow[['avg_snow','max_snow']].round(1)
    nb_out = neighborhoods.merge(nb_snow, left_on=name_col, right_on='neighborhood', how='left')
    nb_out.to_file('outputs/boston_neighborhoods.geojson', driver='GeoJSON')
    print(f'Neighborhoods: {len(nb_out)} areas exported')
else:
    print('Neighborhoods skipped')

Neighborhoods: 25 areas exported


In [79]:
# CELL 19 — Final summary
print('=' * 55)
print('  BOSTON SNOW PIPELINE - COMPLETE')
print('=' * 55)
for path in ['outputs/boston_streets_snow.geojson','outputs/boston_storms.json',
             'outputs/boston_historical.json','outputs/boston_neighborhoods.geojson']:
    if os.path.exists(path):
        mb = os.path.getsize(path)/1e6
        print(f'  OK  {os.path.basename(path):40s} {mb:.1f} MB')
    else:
        print(f'  --  {os.path.basename(path):40s} NOT GENERATED')
print()
print('  Next: upload outputs/ files to Observable')
print('=' * 55)

  BOSTON SNOW PIPELINE - COMPLETE
  OK  boston_streets_snow.geojson              25.4 MB
  OK  boston_storms.json                       0.0 MB
  OK  boston_historical.json                   0.0 MB
  OK  boston_neighborhoods.geojson             0.5 MB

  Next: upload outputs/ files to Observable


---
## Methodology Note

**Data Sources:**
- **CoCoRaHS** — volunteer observer network, daily `NewSnowDepth` readings across MA. Used for current-season street interpolation.
- **NOAA GHCN Daily** — Boston Logan Airport (USW00014739). Used for historical comparisons 1980–2026.
- **OpenStreetMap** — Street geometry via `osmnx`.
- **City of Boston Open Data** — Neighborhood boundaries and plow route classifications.

**Spatial Interpolation:**
Inverse Distance Weighting (IDW), power=2, k=8 nearest neighbors. Each street midpoint is estimated from a weighted average of the 8 nearest CoCoRaHS stations.

**Limitations:**
IDW assumes smooth spatial variation. Plow priority is derived from road classification where city plow-route data is unavailable. CoCoRaHS observers report at variable times so multi-day storms may span observation dates.